# 10 · Simulate some data

**Step 0 of the workflow.** Nothing here involves NanoOrganizer — this is
just a rig writing files, the way a rig does. Run it once; `11` and `12` use
what it leaves behind.

It is deliberately **messy in the way real campaigns are messy**: three
instruments, three folder trees, three naming conventions, none of them
agreeing, and none of them laid out as `<Modality>Data/<SampleID>/`. That is
the point. A convention-based auto-attach cannot find this data, and pretending
otherwise is the thing that makes a demo lie.

```
~/Repos/OrgDemo/Lab/
├── RawData/
│   ├── spectrometer/2026-03-11/      S01_t0000s.csv …   one file per frame
│   ├── microscope_share/S01/         img_01.tif …       a folder per sample
│   └── xrd_rig/                      S01_waxs.dat       one file per sample
└── Meta/
    └── synthesis_dict.json           what the operator wrote down
```

Only the spectra get written into the metadata dict. The micrographs and the
diffraction come from other people on other days, and in `11` they are linked
by hand — which is what actually happens.

**The hidden control variable is the synthesis temperature.** It sets the
particle size, which sets the plasmon band, the diffraction line width and
what the micrographs show. `12` tries to recover it.

The generator itself is `NanoOrganizer/demo/lab.py` — one short function per
instrument, so this notebook runs them one at a time and looks in between. The
web app's **🎓 Demo** page calls the same functions, and so does
`python -m NanoOrganizer.demo --lab`; there is one copy of this rig, not three.

In [1]:
import json

import numpy as np

from NanoOrganizer.demo import lab
from NanoOrganizer.demo.lab import TEMPERATURE_C, band_nm, diameter_nm

paths = lab.lab_paths().make()     # ~/Repos/OrgDemo/Lab, or $NANOORGANIZER_DEMO_ROOT/Lab
LAB, RAW = paths.root, paths.raw

# One random stream for the whole rig, in the order the instruments run, so
# the files come out the same every time.
rng = np.random.default_rng(lab.SEED)

# One hidden number per sample. Everything below follows from it:
#   diameter_nm(T) = 6 + 0.16 (T - 60)      hotter synthesis, bigger particles
#   band_nm(d)     = 512 + 1.9 d            the plasmon band red-shifts as they grow
print(TEMPERATURE_C)
print({s: round(diameter_nm(t), 1) for s, t in TEMPERATURE_C.items()})

{'S01': 60.0, 'S02': 70.0, 'S03': 80.0, 'S04': 90.0, 'S05': 100.0, 'S06': 110.0}
{'S01': 6.0, 'S02': 7.6, 'S03': 9.2, 'S04': 10.8, 'S05': 12.4, 'S06': 14.0}


## The spectrometer

A growth series per sample: fourteen frames over twenty minutes, each a
two-column CSV, with the acquisition time in the filename. The band grows and
red-shifts as the particles do.

In [2]:
written = lab.write_spectra(paths.spectra, TEMPERATURE_C, rng=rng)

print(len(written), "spectra in", paths.spectra)
print(sorted(p.name for p in paths.spectra.glob("S01_*.csv"))[:4])

84 spectra in /home/yuzhang/Repos/OrgDemo/Lab/RawData/spectrometer/2026-03-11
['S01_t0000s.csv', 'S01_t0090s.csv', 'S01_t0180s.csv', 'S01_t0270s.csv']


## The microscope

A folder per sample of calibrated TIFFs, written by whoever was on the
instrument that week. The pixel calibration goes in the image description, so
the micrographs can be read in nanometres later without anyone remembering the
magnification.

In [3]:
written = lab.write_micrographs(paths.scope, TEMPERATURE_C, rng=rng)

# The calibration rides along inside each TIFF, the way the instrument writes it.
print(sorted(p.name for p in (paths.scope / "S01").iterdir()))

['img_01.tif', 'img_02.tif', 'img_03.tif', 'session.txt']


## The diffractometer

One pattern per sample. The line width narrows as the crystallites grow —
Scherrer — so diffraction sees the same hidden number the spectra do, from a
completely different direction.

In [4]:
written = lab.write_diffraction(paths.xrd, TEMPERATURE_C, rng=rng)

print(sorted(p.name for p in paths.xrd.iterdir()))

['S01_waxs.dat', 'S02_waxs.dat', 'S03_waxs.dat', 'S04_waxs.dat', 'S05_waxs.dat', 'S06_waxs.dat']


## What the operator wrote down

The metadata dict: one record per sample, keyed by sample id. It holds the
synthesis conditions and **the spectra only** — the block naming files becomes
a measurement when it is ingested.

The micrographs and the diffraction are *not* in here. Nobody wrote them down,
which is the normal state of affairs and the reason `11` links them by hand.

Saved as JSON so `11` can load the real dict rather than re-running this
notebook.

In [5]:
Synthesis_dict = lab.synthesis_dict(paths.spectra, TEMPERATURE_C)
lab.write_synthesis_dict(paths.synthesis_dict, Synthesis_dict)

print("wrote", paths.synthesis_dict)
Synthesis_dict["S01"]

wrote /home/yuzhang/Repos/OrgDemo/Lab/Meta/synthesis_dict.json


{'sample_id': 'S01',
 'synthesis_batch': {'run_id': 'run_001',
  'batch_tag': '2026-03-11',
  'campaign': 'lab-demo',
  'status': 'done',
  'started_at': '2026-03-11T09:07:00'},
 'conditions': {'temperature_C': 60.0,
  'hold_time_min': 21.0,
  'stir_rate_rpm': 400,
  'solvent': 'water'},
 'reagents': {'precursor': {'concentration_mM': 2.0, 'volume_uL': 500.0},
  'reductant': {'concentration_mM': 16.0, 'volume_uL': 250.0}},
 'uvvis_growth': {'modality': 'uvvis',
  'instrument': 'LabSpec 2000',
  'n_frames': 14,
  'spectrum_glob': '/home/yuzhang/Repos/OrgDemo/Lab/RawData/spectrometer/2026-03-11/S01_t*.csv'}}

## The answer key

Kept beside the data so `12` can check what the pipeline recovers against what
the generator actually used — the one thing real data never lets you do.

In [6]:
truth = lab.lab_truth(TEMPERATURE_C)
truth.to_csv(paths.truth, index=False)
truth

,sample_id,temperature_C,true_diameter_nm,true_band_nm
0,S01,60.0,6.0,523.4
1,S02,70.0,7.6,526.4
2,S03,80.0,9.2,529.5
3,S04,90.0,10.8,532.5
4,S05,100.0,12.4,535.6
5,S06,110.0,14.0,538.6


## What is on disk now

Three trees that do not agree, and a dict that names one of them. Nothing here
is organised yet — that is the next notebook.

In [7]:
from NanoOrganizer import structure

print(structure.tree(RAW, depth=2, limit=4))

📁 RawData  3 folders
├── 📁 microscope_share  6 folders
│   ├── 📁 S01  4 files · .tif 3, .txt 1
│   ├── 📁 S02  4 files · .tif 3, .txt 1
│   ├── 📁 S03  4 files · .tif 3, .txt 1
│   ├── 📁 S04  4 files · .tif 3, .txt 1
│   └── … +2 more  raise the per-layer limit to see them
├── 📁 spectrometer  1 folder
│   └── 📁 2026-03-11  84 files · .csv 84
└── 📁 xrd_rig  6 files · .dat 6
    ├── 📄 S01_waxs.dat  44.0 KB
    ├── 📄 S02_waxs.dat  44.0 KB
    ├── 📄 S03_waxs.dat  44.0 KB
    ├── 📄 S04_waxs.dat  44.0 KB
    └── … +2 more  raise the per-layer limit to see them


---

The data is on disk and nothing knows about it yet. **Next: `11_build_organizer`.**